In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('/home/jovyan/work/base_demo')
import base_tool

In [ ]:
import pandas as pd
import numpy as np
pd.set_option('display.max_rows', 15)
pd.set_option('display.max_columns', 65)

数据介绍

bid_book_begin 集合竞价后的完整委托买入订单簿

ask_book_begin 集合竞价后的完整委托卖出订单簿

snap_list 连续竞价阶段的1s快照
    time_hms  时分秒字符串
    time_mark 毫秒级时间戳
    price_open 快照内首个成交价(无成交时为0.0)
    price_low  快照内最低成交价(无成交时为0.0)
    price_high 快照内最高成交价(无成交时为0.0)
    price_last 当日内最新成交价
     buy_trade 主动买入成交
    sell_trade 主动卖出成交
    bid_insert 委托买入挂单
    ask_insert 委托卖出挂单
    bid_cancel 委托买入撤单
    ask_cancel 委托卖出撤单

In [4]:
instrument_id = '511520'
trade_ymd = '20260319'

In [ ]:
param_dict = {

    'name' : 'TBM',
    'instrument_id' : instrument_id,
    'trade_ymd' : trade_ymd,
    
    'x_window' : 300 ,
    'y_window' : 300 ,
    'stride': 60,

    'k_up': 0.5,
    'k_down': 0.5
}

In [8]:
snap_list = base_tool.snap_list_load(instrument_id, trade_ymd)
print(snap_list[1])

{'time_mark': 1773883801000, 'time_open': 1773883800010, 'time_last': 1773883800430, 'price_open': 116.01, 'price_low': 116.003, 'price_high': 116.01, 'price_last': 116.003, 'price_vwap': 116.0097, 'num_trades': 11, 'num_buy': 0, 'num_sell': 2, 'buy_trade': [], 'sell_trade': [[116.003, 500], [116.01, 600]], 'bid_insert': [[116.007, 5100], [115.909, 6000], [115.908, 6000]], 'ask_insert': [[116.017, 5100], [116.05, 2656], [116.115, 6000], [116.116, 6000]], 'bid_cancel': [[115.852, 11900]], 'ask_cancel': [[116.02, 5500], [116.115, 6000], [116.116, 6000], [116.2, 11900]], 'bid_book': [[116.007, 5100], [116.003, 51500], [116.001, 20000], [115.996, 100], [115.965, 30000]], 'ask_book': [[116.015, 30000], [116.017, 5100], [116.02, 5400], [116.025, 400], [116.026, 400]], 'history': False}


In [ ]:
import numpy as np
import pandas as pd

class TrainValidTest():
    def __init__(self, snap_list, param_dict,feature_func,y_func):
        if param_dict is not None:
            self.__dict__.update(param_dict)
        # 确保必要属性存在
        if not hasattr(self, 'x_window'):
            self.x_window = 1
        if not hasattr(self, 'y_window'):
            self.y_window = 1

        self.snap_list = snap_list.copy()
        self.create_feature = feature_func
        self.create_y = y_func

    def samples(self):
        X_list, y_list = [], []
        n = len(self.snap_list)

        for i in range(self.x_window, n - self.y_window):
            # 特征窗口：[i-x_window, i)
            x = self.create_feature(self.snap_list[i - self.x_window:i])
            # 从特征中提取波动率（基于x窗口计算）
            volatility = x.get('volatility', 0.0)
            # 标签窗口：[i, i+y_window)
            y = self.create_y(self.snap_list[i:i + self.y_window], volatility)
            X_list.append(x)
            y_list.append(y)

        X_all = pd.concat(X_list, axis=0, ignore_index=True)
        y_all = pd.concat(y_list, axis=0, ignore_index=True)

        return X_all, y_all

def create_feature(snap_slice):
    """
    从特征窗口快照切片中提取特征，并计算波动率。
    """
    last = snap_slice[-1]

    price = last['price_last']
    trades = last['num_trades']
    best_bid = last['bid_book'][0][0] if last['bid_book'] else np.nan
    best_ask = last['ask_book'][0][0] if last['ask_book'] else np.nan
    spread = best_ask - best_bid if not np.isnan(best_ask) and not np.isnan(best_bid) else np.nan
    bid_depth = last['bid_book'][0][1] if last['bid_book'] else 0
    ask_depth = last['ask_book'][0][1] if last['ask_book'] else 0

    # 基于特征窗口内的价格序列计算波动率（相对波动率 = 标准差 / 均值）
    prices = [snap['price_last'] for snap in snap_slice if snap['price_last'] is not None]
    if len(prices) >= 2:
        mean_price = np.mean(prices)
        std_price = np.std(prices)
        volatility = std_price / mean_price if mean_price != 0 else 0.0
    else:
        volatility = 0.0

    features = {
        'price_last': price,
        'num_trades': trades,
        'best_bid': best_bid,
        'best_ask': best_ask,
        'spread': spread,
        'bid_depth1': bid_depth,
        'ask_depth1': ask_depth,
        'volatility': volatility,   # 波动率作为特征之一
    }
    return pd.Series(features)

def create_y(snap_slice, volatility):
    """
    根据标签窗口快照切片和波动率（来自特征窗口）计算标签。
    """
    up_factor = 1.25
    down_factor = 0.8

    first_price = snap_slice[0]['price_last']
    if first_price is None:
        return 0

    # 上下轨基于特征窗口波动率计算
    up = first_price * (1 + volatility * up_factor)
    down = first_price * (1 - volatility * down_factor)

    t_up = len(snap_slice)
    t_down = len(snap_slice)

    for i, now in enumerate(snap_slice):
        price = now['price_last']
        if price is None:
            continue
        if price > up and t_up == len(snap_slice):
            t_up = i
        if price < down and t_down == len(snap_slice):
            t_down = i

    if t_up < t_down:
        return 1
    elif t_up > t_down:
        return -1
    else:
        return 0

In [ ]:
%%time
from collections import deque
import logging
import os

class StrategyDemo():
    def __init__(self, param_dict={}) -> None:
        # self.model = joblib.load(file)
        self.__dict__.update(param_dict)
        data_file = f'/home/jovyan/work/backtest_result/{self.instrument_id}_{self.trade_ymd}_{self.name}.pkl' 
        if os.path.exists(data_file):
            os.remove(data_file)

        self.position_last = 0
        self.price_list = deque(maxlen=self.long_window)
        self.prev_signal = 0


        return

    def on_snap(self, snap:dict) -> None:
        price = snap['price_last']

        if price == 0.0 or price == None:
            return

        self.price_list.append(price)

        if(len(self.price_list) < self.long_window):
            self.position_last = 0
            self.prev_signal = 0
            return

        short_ma = sum(list(self.price_list)[-self.short_window:]) / self.short_window
        long_ma = sum(self.price_list) / self.long_window
        diff = short_ma - long_ma

        if diff > self.threshold:
            current_signal = 1
        elif diff < -self.threshold:
            current_signal = -1
        else :
            current_signal = 0

        if current_signal != self.prev_signal:
            self.position_last = current_signal
            self.prev_signal = current_signal

        return


